In [1]:
import pandas as pd
import numpy as np
from transformers import BertModel, BertTokenizer, BertForSequenceClassification
import torch
from datetime import datetime
import os
import json

/home/guicbelon/Documentos/IC/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TICKERS_TEXTS = {
    'BA': ['Boeing', 'Dreamliner', 'Boeing Defense'],
    'MCD': ['McDonald\'s', 'McFlurry', 'Big Mac', 'McCafé'],
    'JNJ': ['Johnson & Johnson', 'Tylenol', 'Neutrogena', 'Janssen Pharmaceuticals'],
    'INTC': ['Intel', 'Pentium', 'Xeon', 'Intel Inside'],
    'BAC': ['Bank of America', 'Merrill Lynch', 'Better Money Habits', 'BankAmeriDeals'],
    'AMZN': ['Amazon', 'Prime Video', 'AWS', 'Alexa'],
    'NVDA': ['NVIDIA', 'GeForce', 'CUDA', 'Tegra'],
    'ORCL': ['Oracle', 'Oracle Database', 'Exadata', 'Java Platform'],
    'DIS': ['Disney', 'Disneyland', 'ESPN', 'Pixar'],
    'GS': ['Goldman Sachs', 'Marcus', '10,000 Small Businesses', 'Goldman Sachs Research'],
    'GOOG': ['Google', 'AdSense', 'YouTube', 'PageRank'],
    'MSFT': ['Microsoft', 'Windows', 'Xbox', 'Azure'],
    'TSLA': ['Tesla', 'Autopilot', 'Cybertruck', 'Gigafactory'],
    'META': ['Meta', 'WhatsApp', 'Instagram', 'Oculus'],
    'AAPL': ['Apple', 'iOS', 'Apple Watch', 'MacBook'],
    'V': ['Visa', 'VisaNet', 'Visa Direct', 'Cybersource'],
    'PFE': ['Pfizer', 'Comirnaty', 'Prevnar', 'Pfizer Oncology'],
    'KO': ['Coca-Cola', 'Minute Maid', 'Powerade'],
    'GE': ['General Electric', 'Predix', 'GE Renewable Energy', 'Additive Manufacturing'],
    'CAT': ['Caterpillar', 'Cat®', 'Track-Type Tractor', 'Cat Financial'],
    'XOM': ['Exxon Mobil', 'Mobil 1', 'Esso', 'ExxonMobil Chemical'],
    'CVX': ['Chevron', 'Chevron Phillips', 'Chevron Texaco', 'Delo'],
    'DE': ['Deere', 'GreenStar', 'S-Series Combine', 'John Deere Financial'],
    'F': ['Ford', 'F-150', 'Lincoln', 'EcoBoost'],
    'WMT': ['Walmart', 'Great Value', 'Sam\'s Club', 'Walmart+']
}


In [3]:
class NLPInfo():
    def __init__(self, 
                 bert_model:str = 'bert-large-uncased',
                 finbert_model:str = 'yiyanghkust/finbert-tone'):
        self.bert_model = BertModel.from_pretrained(bert_model)
        self.bert_tokenizer = BertTokenizer.from_pretrained(bert_model)
        self.finbert_model = BertForSequenceClassification.from_pretrained(finbert_model)
        self.finbert_tokenizer = BertTokenizer.from_pretrained(finbert_model)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.bert_model.to(self.device)
        self.finbert_model.to(self.device)
        self.bert_model.eval()
        self.finbert_model.eval()
        
    def create_reference_embeddings(self, ticker:str):
        texts = TICKERS_TEXTS[ticker] + [ticker]
        self.reference_embeddings = []
        for text in texts:
            self.reference_embeddings.append(self.create_bert_embedding(text))
        
    def create_bert_embedding(self, text:str):
        with torch.no_grad():
            input_ids = self.bert_tokenizer.encode(text, max_length=512, truncation=True, return_tensors="pt").to(self.device)
            output = self.bert_model(input_ids)
            return output[0].mean(dim=1).cpu().numpy()
        
    def create_finbert_tensor(self, text:str):
        with torch.no_grad():
            input_ids = self.finbert_tokenizer.encode(text, max_length=512, truncation=True, return_tensors="pt").to(self.device)
            output = self.finbert_model(input_ids)
            return output.logits.cpu().numpy()
    
    def calculate_best_euclidean_distance(self, new_embedding:np.array):
        best_distance = np.inf
        for ref_embedding in self.reference_embeddings:
            distance = np.linalg.norm(new_embedding - ref_embedding)
            if distance < best_distance:
                best_distance = distance
        return best_distance
        
    def create_mlp_df_from_news(self, news_df:pd.DataFrame, temp_df=None, save_count:int=35):
        dfs_to_concat = []        
        ticker = news_df['ticker'].iloc[0]
        self.create_reference_embeddings(ticker)
        if temp_df is not None:
            news_df = news_df.loc[news_df.index > temp_df.index[-1]]    
            dfs_to_concat.append(temp_df)
            print("Tamanho do dataframe temporário: ",len(temp_df))
        count = 0
        for date in news_df.index:
            new_info = news_df.loc[date]
            texts_embedding_distance = []
            finbert_tensors = []
            new_text = new_info['news']
            if not pd.isna(new_text):
                text_split = new_text.split("\n")
                for text in text_split:
                    bert_embedding = self.create_bert_embedding(text)
                    euclidean_distance = self.calculate_best_euclidean_distance(bert_embedding)
                    finbert_tensor = self.create_finbert_tensor(text)
                    finbert_tensor_str = json.dumps((finbert_tensor[0]).tolist()) 
                    texts_embedding_distance.append(euclidean_distance)
                    finbert_tensors.append(finbert_tensor_str)
            df_to_add = pd.DataFrame({'date': date,
                               'ticker': ticker,                                   
                               'embedding_distance': texts_embedding_distance,
                               'finbert': finbert_tensors})
            df_to_add.set_index('date', inplace=True)
            dfs_to_concat.append(df_to_add)
            count +=1
            if count >= save_count:
                temp_df = pd.concat(dfs_to_concat)
                temp_df.to_csv(f"temps_nlp/{ticker}.csv")
                print(f"Last updated for {ticker}: {datetime.now()}")
                count = 0            
        return pd.concat(dfs_to_concat)

        
        
nlp_info = NLPInfo()

In [4]:

folder = 'news'
files = os.listdir(folder)
tickers_raw = [f.split("_")[0] for f in files if os.path.isfile(os.path.join(folder, f))]

folder_ebd = 'ebd_fbrt'
files = os.listdir(folder_ebd)
tickers_to_remove = [f.split("_")[0] for f in files if os.path.isfile(os.path.join(folder_ebd, f))]
tickers = [ticker for ticker in tickers_raw if ticker not in tickers_to_remove]
print(tickers)

['TSLA', 'META', 'AAPL', 'V', 'PFE', 'KO', 'GE', 'CAT', 'XOM', 'CVX', 'DE', 'F', 'WMT']


In [5]:
for ticker in tickers:
    print("\nCreating NLP for", ticker)
    df = pd.read_csv(f'full_texts/{ticker}_full_texts_2024-04-01-2024-09-01.csv', index_col=0)
    df.index = pd.to_datetime(df.index)
    temp_df = None
    start_date = None
    try:
        temp_df = pd.read_csv(f"temps_nlp/{ticker}.csv", index_col=0)
        temp_df.index = pd.to_datetime(temp_df.index)
        start_date = temp_df.index[-1]
    except:pass
    print(f"start_date: {start_date}")
    
    df_nlp = nlp_info.create_mlp_df_from_news(df, temp_df)
    df_nlp.to_csv(f'ebd_fbrt/{ticker}_ebd_fbrt_2024-04-01-2024-09-01.csv', index=True)
    


Creating NLP for TSLA
start_date: 2024-04-04 04:13:31
Tamanho do dataframe temporário:  7112
Last updated for TSLA: 2024-11-16 20:54:11.957964
Last updated for TSLA: 2024-11-16 21:03:54.829952
Last updated for TSLA: 2024-11-16 21:08:47.745510
Last updated for TSLA: 2024-11-16 21:14:53.707473
Last updated for TSLA: 2024-11-16 21:21:28.973797
Last updated for TSLA: 2024-11-16 21:28:26.337993
Last updated for TSLA: 2024-11-16 21:35:43.494339
Last updated for TSLA: 2024-11-16 21:40:10.569875
Last updated for TSLA: 2024-11-16 21:47:16.063950
Last updated for TSLA: 2024-11-16 21:52:10.902399
Last updated for TSLA: 2024-11-16 21:52:11.021518
Last updated for TSLA: 2024-11-16 21:52:11.129589
Last updated for TSLA: 2024-11-16 21:52:11.254598
Last updated for TSLA: 2024-11-16 21:52:11.364771
Last updated for TSLA: 2024-11-16 21:53:31.826440
Last updated for TSLA: 2024-11-16 22:00:03.279554
Last updated for TSLA: 2024-11-16 22:06:21.123853
Last updated for TSLA: 2024-11-16 22:12:30.213619
Last u

In [9]:
string_vetor = temp_df.embedding[0]
string_vetor

/tmp/ipykernel_12043/3843075135.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  string_vetor = temp_df.embedding[0]


'[-0.4013297 , 0.13111034,-0.28810847,...,-0.44735223,-0.3087233 ,\n  0.07619484]'

In [11]:
ebd = nlp_info.create_bert_embedding("I am a robot")

In [26]:
ts = nlp_info.create_finbert_tensor("I am a robot")
ts

array([[ 3.4072707 , -4.8622947 , -0.56910855]], dtype=float32)

In [27]:
import numpy as np

# Cria o vetor
vetor = ts[0]

# Converte para string com alta precisão
string_vetor = np.array2string(vetor, precision=9, separator=',')
print(string_vetor, len(string_vetor))  


[ 3.4072707 ,-4.8622947 ,-0.56910855] 37


In [28]:
import json

# Serializa o vetor para JSON
string_vetor = json.dumps(vetor.tolist())  # Converte para lista e depois serializa
print(len(string_vetor))

60


In [13]:
df.index[:6]

DatetimeIndex(['2024-04-01 05:09:24', '2024-04-01 07:10:00',
               '2024-04-01 09:58:00', '2024-04-01 11:13:00',
               '2024-04-01 12:15:53', '2024-04-01 13:15:00'],
              dtype='datetime64[ns]', name='time', freq=None)

In [10]:
aa = pd.read_csv("temps_nlp/BA.csv", index_col=0)


Index(['2024-04-01 05:09:24', '2024-04-01 07:10:00', '2024-04-01 09:58:00',
       '2024-04-01 11:13:00', '2024-04-01 12:15:53'],
      dtype='object', name='date')

In [3]:
a = nlp_info.create_finbert_embedding('hello world')


In [8]:
import numpy as np

# Cria o vetor
vetor = np.array([0.123456789, 1.23456789, 2.3456789])

# Converte para string com alta precisão
string_vetor = np.array2string(vetor, precision=5, separator=',', suppress_small=False)
print(string_vetor)


[0.12346,1.23457,2.34568]


In [9]:
string_vetor

'[0.12346,1.23457,2.34568]'